In [1]:
import re
import csv
import os

input_file = 'mat_feat_dlmc.txt'  # Ensure this matches your log filename

# Regex to find the sparsity in the file path (looks for /number/ before the filename)
path_pattern = re.compile(r'/(\d+\.\d+)/body_decoder')
# Regex to find the data line
data_pattern = re.compile(r'^(\d+\.\d+)\s+\[\'(.*?)\'\]=\'(.*)\'')

# Dictionary to hold lists of rows grouped by sparsity { "0.6": [ [...], [...] ] }
grouped_data = {}

headers = [
    "path_sparsity", "mem_footprint", "title_base", "m", "n", "nnz", 
    "computed_sparsity", "nnz_per_row_avg", "nnz_per_row_std", 
    "bw_avg_n", "nnz_max_diff", "num_neigh_avg", "cross_row_similarity_avg"
]

current_sparsity = "unknown"

with open(input_file, 'r') as f:
    for line in f:
        line = line.strip()
        
        # 1. Check if this line contains the file path to update the current sparsity
        path_match = path_pattern.search(line)
        if path_match:
            current_sparsity = path_match.group(1)
            continue
            
        # 2. Check if this is the data line
        data_match = data_pattern.match(line)
        if data_match:
            mem_footprint = data_match.group(1)
            title = data_match.group(2)
            values = data_match.group(3).split()
            
            # Filter and map values based on your fprintf sequence
            # We skip index 6 ("normal"), 7 ("random"), and 12 ("14")
            row = [
                current_sparsity,
                mem_footprint,
                title,
                values[0],  # m
                values[1],  # n
                values[2],  # nnz
                values[3],  # sparsity (from calculation)
                values[4],  # nnz_per_row_avg
                values[5],  # nnz_per_row_std
                values[8],  # bw_avg / n
                values[9],  # (nnz_max - avg) / avg
                values[10], # num_neigh_avg
                values[11]  # cross_row_similarity_avg
            ]
            
            if current_sparsity not in grouped_data:
                grouped_data[current_sparsity] = []
            grouped_data[current_sparsity].append(row)

# 3. Write one CSV file for each sparsity found
for sparsity, rows in grouped_data.items():
    filename = f"dlmc_matrix_stats_sparsity_{sparsity}.csv"
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(headers)
        writer.writerows(rows)
    print(f"Created: {filename} with {len(rows)} entries.")

Created: dlmc_matrix_stats_sparsity_0.5.csv with 68 entries.
Created: dlmc_matrix_stats_sparsity_0.6.csv with 68 entries.
Created: dlmc_matrix_stats_sparsity_0.7.csv with 68 entries.
Created: dlmc_matrix_stats_sparsity_0.8.csv with 68 entries.
Created: dlmc_matrix_stats_sparsity_0.9.csv with 68 entries.
Created: dlmc_matrix_stats_sparsity_0.95.csv with 68 entries.
Created: dlmc_matrix_stats_sparsity_0.98.csv with 62 entries.


In [5]:
import re
import csv
import os
from collections import defaultdict

input_file = '../results_4_3_26_dlmc.txt'  # Replace with your actual filename

# --- CONFIGURATION ---
KEEP_FORMATS = ["MKL_IE", "AOCL", "COO_RowSplit_Vec", "COO_Z_Seg_NoAtomic", "COO_Hilbert_Seg_NoAtomic", "COO_ColInd0_Vec"] 
KEEP_K_VALUES = ["256"]
# ---------------------

# Regex to find sparsity AND filename from the performance log line
# Example: .../0.6/body_decoder_layer_1.smtx
path_name_pattern = re.compile(r'/(\d+\.\d+)/([^/]+)\.smtx', re.IGNORECASE)
data_pattern = re.compile(r'format:\s*([^,]+).*?k:\s*(\d+).*?gflops:\s*(\d+\.\d+)', re.IGNORECASE)

# Nested dictionary: grouped_perf[sparsity][matrix_name][column] = gflops
grouped_perf = defaultdict(lambda: defaultdict(dict))
all_columns = set()

current_matrix = None
current_sparsity = None

with open(input_file, 'r') as f:
    for line in f:
        line = line.strip()
        if not line: continue

        # 1. Update both Sparsity and Matrix Name from the path
        path_match = path_name_pattern.search(line)
        if path_match:
            current_sparsity = path_match.group(1)
            current_matrix = path_match.group(2)
        
        # 2. Extract performance metrics
        data_match = data_pattern.search(line)
        if data_match and current_matrix and current_sparsity:
            fmt = data_match.group(1).strip()
            k_val = data_match.group(2).strip()
            gflops = data_match.group(3).strip()

            if KEEP_FORMATS and fmt not in KEEP_FORMATS: continue
            if KEEP_K_VALUES and k_val not in KEEP_K_VALUES: continue

            col_name = f"GFLOPS-{fmt}-k={k_val}"
            all_columns.add((fmt, int(k_val), col_name))
            grouped_perf[current_sparsity][current_matrix][col_name] = gflops

# 3. Write one performance CSV for each sparsity found
sorted_col_tuples = sorted(list(all_columns), key=lambda x: (x[0], x[1]))
sorted_headers = [col[2] for col in sorted_col_tuples]
headers = ['title_base'] + sorted_headers

for sparsity, matrices in grouped_perf.items():
    out_name = f"dlmc_perf_sparsity_{sparsity}.csv"
    with open(out_name, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        for matrix_name in sorted(matrices.keys()):
            perf_metrics = matrices[matrix_name]
            row = [matrix_name] + [perf_metrics.get(col, "") for col in sorted_headers]
            writer.writerow(row)
    print(f"Created: {out_name} with {len(matrices)} matrices.")

Created: dlmc_perf_sparsity_0.5.csv with 66 matrices.
Created: dlmc_perf_sparsity_0.6.csv with 66 matrices.
Created: dlmc_perf_sparsity_0.7.csv with 66 matrices.
Created: dlmc_perf_sparsity_0.8.csv with 66 matrices.
Created: dlmc_perf_sparsity_0.9.csv with 66 matrices.
Created: dlmc_perf_sparsity_0.95.csv with 66 matrices.
Created: dlmc_perf_sparsity_0.98.csv with 60 matrices.


In [6]:
import pandas as pd
import glob

# Find all sparsity feature files
feature_files = glob.glob('dlmc_matrix_stats_sparsity_*.csv')

for f_file in feature_files:
    # Extract the sparsity value from the filename (e.g., 0.6)
    sparsity_val = f_file.split('_')[-1].replace('.csv', '')
    p_file = f"dlmc_perf_sparsity_{sparsity_val}.csv"
    
    if os.path.exists(p_file):
        df_feat = pd.read_csv(f_file)
        df_perf = pd.read_csv(p_file)
        
        # Merge on title_base
        final_df = pd.merge(df_feat, df_perf, on='title_base', how='inner')
        
        final_output = f"DLMC_FULL_DATA_sparsity_{sparsity_val}.csv"
        final_df.to_csv(final_output, index=False)
        print(f"Successfully combined sparsity {sparsity_val} into {final_output}")
    else:
        print(f"Warning: No performance file found for sparsity {sparsity_val}")

Successfully combined sparsity 0.98 into DLMC_FULL_DATA_sparsity_0.98.csv
Successfully combined sparsity 0.95 into DLMC_FULL_DATA_sparsity_0.95.csv
Successfully combined sparsity 0.9 into DLMC_FULL_DATA_sparsity_0.9.csv
Successfully combined sparsity 0.7 into DLMC_FULL_DATA_sparsity_0.7.csv
Successfully combined sparsity 0.5 into DLMC_FULL_DATA_sparsity_0.5.csv
Successfully combined sparsity 0.6 into DLMC_FULL_DATA_sparsity_0.6.csv
Successfully combined sparsity 0.8 into DLMC_FULL_DATA_sparsity_0.8.csv


In [7]:
import re
import csv

input_file = 'mat_feat_graph.txt'  # Replace with your actual filename
output_file = 'matrix_features_graph.csv'

# Define the header based on your C code
headers = [
    "mem_footprint", "title_base", "m", "n", "nnz", "sparsity", 
    "nnz_per_row_avg", "nnz_per_row_std", "bw_avg_n", 
    "nnz_max_diff", "num_neigh_avg", "cross_row_similarity_avg"
]

data_rows = []

# Pattern to match the lines starting with the memory footprint value
# e.g., 1.30203 ['title']=...
pattern = re.compile(r'^(\d+\.\d+)\s+\[\'(.*?)\'\]=\'(.*)\'')

with open(input_file, 'r') as f:
    for line in f:
        match = pattern.match(line.strip())
        if match:
            mem_footprint = match.group(1)
            title = match.group(2)
            # Split the space-separated values inside the quotes
            values = match.group(3).split()
            
            # The values array currently contains:
            # [m, n, nnz, sparsity, avg, std, "normal", "random", bw, max_diff, neigh, similarity, "14", title]
            
            # We filter out "normal", "random", and "14" by index
            filtered_values = [

                mem_footprint,
                title,
                values[0],  # m
                values[1],  # n
                values[2],  # nnz
                values[3],  # sparsity
                values[4],  # nnz_per_row_avg
                values[5],  # nnz_per_row_std
                values[8],  # bw_avg / n
                values[9],  # nnz_max_diff
                values[10], # num_neigh_avg
                values[11]  # cross_row_similarity_avg
            ]
            data_rows.append(filtered_values)

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    writer.writerows(data_rows)

print(f"Successfully created {output_file}")

Successfully created matrix_features_graph.csv


In [8]:
import re
import csv
from collections import defaultdict

input_file = '../results_4_3_26_graph.txt'  # Replace with your actual filename
output_file = 'graph_matrix_performance.csv'

# --- CONFIGURATION AREA ---
# Leave these as empty lists [] to keep EVERYTHING.
# Otherwise, specify exactly what you want, e.g., ["MKL_IE", "AOCL"]
KEEP_FORMATS = ["MKL_IE", "AOCL", "COO_RowSplit_Vec", "COO_Z_Seg_NoAtomic", "COO_Hilbert_Seg_NoAtomic", "COO_ColInd0_Vec"] 
KEEP_K_VALUES = ["256"] # e.g., ["8", "16"]
# ---------------------------

matrix_header_pattern = re.compile(r'/([^/]+)\.mtx')
data_pattern = re.compile(r'format:\s*([^,]+).*?k:\s*(\d+).*?gflops:\s*(\d+\.\d+)', re.IGNORECASE)

matrix_data = defaultdict(dict)
all_columns = set()
current_matrix = None

with open(input_file, 'r') as f:
    for line in f:
        line = line.strip()
        if not line: continue
            
        if line.startswith('./') and 'SpMM' not in line:
            name_match = matrix_header_pattern.search(line)
            if name_match:
                current_matrix = name_match.group(1)
                continue

        data_match = data_pattern.search(line)
        if data_match and current_matrix:
            fmt = data_match.group(1).strip()
            k_val = data_match.group(2).strip()
            gflops = data_match.group(3).strip()
            
            # Apply Filters
            if KEEP_FORMATS and fmt not in KEEP_FORMATS:
                continue
            if KEEP_K_VALUES and k_val not in KEEP_K_VALUES:
                continue

            col_name = f"GFLOPS-{fmt}-k={k_val}"
            all_columns.add((fmt, int(k_val), col_name)) # Store as tuple for sorting
            matrix_data[current_matrix][col_name] = gflops

if not matrix_data:
    print("No data found matching your filters.")
else:
    # Sort columns: First by Format name, then Numerically by K
    sorted_col_tuples = sorted(list(all_columns), key=lambda x: (x[0], x[1]))
    sorted_headers = [col[2] for col in sorted_col_tuples]
    
    headers = ['title_base'] + sorted_headers

    with open(output_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        
        for matrix in sorted(matrix_data.keys()):
            perf_metrics = matrix_data[matrix]
            row = [matrix] + [perf_metrics.get(col, "") for col in sorted_headers]
            writer.writerow(row)

    print(f"Successfully created {output_file} with {len(matrix_data)} matrices.")

Successfully created graph_matrix_performance.csv with 18 matrices.


In [9]:
import pandas as pd

# Load both files
df_features = pd.read_csv('matrix_features_graph.csv')
df_performance = pd.read_csv('graph_matrix_performance.csv')

# Merge on the matrix name column
# Note: Ensure the names in 'title_base' match exactly (e.g., 'citeseer')
final_df = pd.merge(df_features, df_performance, on='title_base', how='inner')

# Save the combined result
final_df.to_csv('combined_matrix_data.csv', index=False)